# Experiments Guide

## 0. Colab Setup

In [ ]:
!git clone -b dev-explainable-clustering-for-prototypes-IEMOCAP https://github.com/luigiaceto/explainable-models-for-speech-analysis.git
%cd explainable-models-for-speech-analysis
%pip install -r requirements-colab.txt

## 1. Project Setup

In [1]:
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd().resolve()
while not (PROJECT_ROOT / "src").exists() and PROJECT_ROOT.parent != PROJECT_ROOT:
    PROJECT_ROOT = PROJECT_ROOT.parent

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.utils.audio_features import pooled_feature_dim
from src.utils.naming import (
    model_name_to_slug,
    pooled_feature_dir_name
)

DATASET_ID = "iemocap_speech_4class"
DATA_DIR = PROJECT_ROOT / "data"
RAW_DIR = DATA_DIR / "raw" / DATASET_ID
AUDIO_DIR = RAW_DIR / "audio"

# (model_name, dimension_of_the_original_vectors_generated_by_the_model)
FEATURE_EXTRACTOR = ("microsoft/wavlm-large", 1024)

FEATURE_EXTRACTOR_NAME, ENCODER_EMBEDDING_DIM = FEATURE_EXTRACTOR
FEATURE_EXTRACTOR_ID = model_name_to_slug(FEATURE_EXTRACTOR_NAME)
FEATURE_POOLING = "mean_std"
FEATURE_DIM = pooled_feature_dim(ENCODER_EMBEDDING_DIM, FEATURE_POOLING)
MIN_DURATION_SECONDS = 2.0
MAX_DURATION_SECONDS = 15.0
DURATION_FILTER_ID = f"dur_gt_{MIN_DURATION_SECONDS:g}_le_{MAX_DURATION_SECONDS:g}"

FEATURE_RUN_ID = f"{DATASET_ID}_{pooled_feature_dir_name(FEATURE_EXTRACTOR_NAME, FEATURE_POOLING)}_{DURATION_FILTER_ID}"
BLACK_BOX_RUN_ID = f"blackbox_{DATASET_ID}_{FEATURE_EXTRACTOR_ID}_{DURATION_FILTER_ID}"
BLACK_BOX_EMBEDDING_RUN_ID = f"{BLACK_BOX_RUN_ID}_penultimate_l2"

FEATURE_DIR = DATA_DIR / "features" / FEATURE_RUN_ID
BLACK_BOX_CHECKPOINT_DIR = PROJECT_ROOT / "checkpoints" / BLACK_BOX_RUN_ID
BLACK_BOX_REPORT_DIR = PROJECT_ROOT / "reports" / BLACK_BOX_RUN_ID
BLACK_BOX_EMBEDDING_DIR = DATA_DIR / "features" / BLACK_BOX_EMBEDDING_RUN_ID

FEATURE_EXTRACTION_BATCH_SIZE = 8
FEATURE_EXTRACTION_NUM_WORKERS = 0
RANDOM_STATE = 42

PROTOTYPE_CLUSTERING_RUN_ID = f"prototype_clustering_{DATASET_ID}_{FEATURE_EXTRACTOR_ID}_{DURATION_FILTER_ID}"
PROTOTYPE_CLUSTERING_CHECKPOINT_DIR = PROJECT_ROOT / "checkpoints" / PROTOTYPE_CLUSTERING_RUN_ID
PROTOTYPE_CLUSTERING_REPORT_DIR = PROJECT_ROOT / "reports" / PROTOTYPE_CLUSTERING_RUN_ID

PROJECT_ROOT

PosixPath('/Users/luigi/Documents/MSc-PoliTO/II_anno/XAI/progetto/explainable-models-for-speech-analysis')

## 2. Download IEMOCAP Speech

In [2]:
from src.preprocessing.download_iemocap import download_iemocap

metadata = download_iemocap(
    output_dir=RAW_DIR,
    overwrite=False
)

metadata.head(n=10)

Generating Session1 split:   0%|          | 0/1085 [00:00<?, ? examples/s]

Generating Session2 split:   0%|          | 0/1023 [00:00<?, ? examples/s]

Generating Session3 split:   0%|          | 0/1151 [00:00<?, ? examples/s]

Generating Session4 split:   0%|          | 0/1031 [00:00<?, ? examples/s]

Generating Session5 split:   0%|          | 0/1241 [00:00<?, ? examples/s]

...

Writing IEMOCAP WAV files (Session2):   0%|          | 0/1023 [00:00<?, ?it/s]

Writing IEMOCAP WAV files (Session3):   0%|          | 0/1151 [00:00<?, ?it/s]

Writing IEMOCAP WAV files (Session4):   0%|          | 0/1031 [00:00<?, ?it/s]

Writing IEMOCAP WAV files (Session5):   0%|          | 0/1241 [00:00<?, ?it/s]

,file_name,source_split,session_id,emotion,label,audio_path,duration_seconds,hf_dataset,hf_split,hf_index,hf_emotion,source_audio_file
0,Session1_00000.wav,Session1,Ses01,sad,3,/Users/luigi/Documents/MSc-PoliTO/II_anno/XAI/...,9.331188,tarasabkar/IEMOCAP_Speech,Session1,0,sad,
1,Session1_00001.wav,Session1,Ses01,sad,3,/Users/luigi/Documents/MSc-PoliTO/II_anno/XAI/...,2.930000,tarasabkar/IEMOCAP_Speech,Session1,1,sad,
2,Session1_00002.wav,Session1,Ses01,sad,3,/Users/luigi/Documents/MSc-PoliTO/II_anno/XAI/...,5.723500,tarasabkar/IEMOCAP_Speech,Session1,2,sad,
3,Session1_00003.wav,Session1,Ses01,neutral,2,/Users/luigi/Documents/MSc-PoliTO/II_anno/XAI/...,5.652500,tarasabkar/IEMOCAP_Speech,Session1,3,neu,
4,Session1_00004.wav,Session1,Ses01,sad,3,/Users/luigi/Documents/MSc-PoliTO/II_anno/XAI/...,5.769938,tarasabkar/IEMOCAP_Speech,Session1,4,sad,
5,Session1_00005.wav,Session1,Ses01,sad,3,/Users/luigi/Documents/MSc-PoliTO/II_anno/XAI/...,2.560000,tarasabkar/IEMOCAP_Speech,Session1,5,sad,
6,Session1_00006.wav,Session1,Ses01,neutral,2,/Users/luigi/Documents/MSc-PoliTO/II_anno/XAI/...,4.215000,tarasabkar/IEMOCAP_Speech,Session1,6,neu,
7,Session1_00007.wav,Session1,Ses01,sad,3,/Users/luigi/Documents/MSc-PoliTO/II_anno/XAI/...,3.886813,tarasabkar/IEMOCAP_Speech,Session1,7,sad,
8,Session1_00008.wav,Session1,Ses01,sad,3,/Users/luigi/Documents/MSc-PoliTO/II_anno/XAI/...,4.250000,tarasabkar/IEMOCAP_Speech,Session1,8,sad,
9,Session1_00009.wav,Session1,Ses01,sad,3,/Users/luigi/Documents/MSc-PoliTO/II_anno/XAI/...,3.030500,tarasabkar/IEMOCAP_Speech,Session1,9,sad,


## 3. Extract Frozen Audio Encoder Features

In [3]:
from src.preprocessing.extract_audio_features import extract_audio_features

feature_paths = extract_audio_features(
    metadata_csv=RAW_DIR / "metadata.csv",
    audio_dir=AUDIO_DIR,
    output_dir=FEATURE_DIR,
    model_name=FEATURE_EXTRACTOR_NAME,
    expected_encoder_embedding_dim=ENCODER_EMBEDDING_DIM,
    pooling=FEATURE_POOLING,
    batch_size=FEATURE_EXTRACTION_BATCH_SIZE,
    num_workers=FEATURE_EXTRACTION_NUM_WORKERS,
    min_duration_seconds=MIN_DURATION_SECONDS,
    max_duration_seconds=MAX_DURATION_SECONDS,
    overwrite=False
)

feature_paths

Loading weights:   0%|          | 0/488 [00:00<?, ?it/s]

Extracting wavlm_large features:   0%|          | 0/564 [00:00<?, ?it/s]

/opt/miniconda3/envs/xai_project/lib/python3.12/site-packages/torch/nn/functional.py:6441: UserWarning: Support for mismatched key_padding_mask and attn_mask is deprecated. Use same type for both instead.
  key_padding_mask = _canonical_mask(


KeyboardInterrupt: 

## 4. Dataset Statistics

In [ ]:
from src.data.iemocap import load_features, print_dataset_statistics

_, feature_metadata = load_features(FEATURE_DIR, mmap_mode="r")
print_dataset_statistics(feature_metadata)

## 5. Train the Black-Box Classifier

In [ ]:
from src.data.iemocap import EMOTION_NAMES
from src.training.train_blackbox import TrainingConfig, train_blackbox

SPLIT_STRATEGY = "sample_stratified"
#SPLIT_STRATEGY = "speaker_independent" # with this mirror, this is session-independent

training_config = TrainingConfig(
    input_dim=FEATURE_DIM,
    feature_extractor_name=FEATURE_EXTRACTOR_NAME,
    encoder_embedding_dim=ENCODER_EMBEDDING_DIM,
    pooling=FEATURE_POOLING,
    hidden_dims=(256, 128),
    num_classes=len(EMOTION_NAMES),
    batch_size=64,
    epochs=100,
    dropout=0.25,
    learning_rate=2e-4,
    weight_decay=5e-4,
    split_strategy=SPLIT_STRATEGY,
    speaker_column="session_id",
    early_stopping_patience=15,
    #lr_scheduler=None, # set explicitly to None to disable the default scheduler
    scheduler_patience=6,
    random_state=RANDOM_STATE
)

training_results = train_blackbox(
    feature_dir=FEATURE_DIR,
    output_dir=BLACK_BOX_CHECKPOINT_DIR,
    config=training_config
)

## 6. Evaluate the Black-Box Classifier

In [ ]:
from src.evaluation.evaluate_blackbox import evaluate_blackbox
from src.evaluation.metrics import print_classification_metrics

test_metrics = evaluate_blackbox(
    checkpoint_path=BLACK_BOX_CHECKPOINT_DIR / "best_model.pt",
    feature_dir=FEATURE_DIR,
    splits_csv=BLACK_BOX_CHECKPOINT_DIR / "splits.csv",
    split="test",
    output_dir=BLACK_BOX_REPORT_DIR
)

print_classification_metrics(test_metrics)

In [ ]:
from IPython.display import Image, display

confusion_matrix_path = BLACK_BOX_REPORT_DIR / "test_confusion_matrix.png"
if confusion_matrix_path.exists():
    display(Image(filename=str(confusion_matrix_path)))

## 7. Visualize Embedding Spaces

Project the pooled audio encoder features and the trained black-box penultimate representations to two PCA dimensions.


In [ ]:
from src.utils.visualize import plot_blackbox_embedding_pca

split_to_visualize = "all"

embedding_pca_result = plot_blackbox_embedding_pca(
    feature_dir=FEATURE_DIR,
    checkpoint_path=BLACK_BOX_CHECKPOINT_DIR / "best_model.pt",
    splits_csv=BLACK_BOX_CHECKPOINT_DIR / "splits.csv",
    split=split_to_visualize,
    output_path=BLACK_BOX_REPORT_DIR / f"{split_to_visualize}_embedding_pca.png",
    random_state=RANDOM_STATE
)

embedding_pca_result["output_path"]

## 8. Extract Black-Box Penultimate Embeddings

Extract the 128-dimensional representation before the final black-box classification layer and save its L2-normalized version.

In [ ]:
from src.preprocessing.extract_blackbox_embeddings import extract_blackbox_penultimate_embeddings_l2

blackbox_embedding_paths = extract_blackbox_penultimate_embeddings_l2(
    feature_dir=FEATURE_DIR,
    checkpoint_path=BLACK_BOX_CHECKPOINT_DIR / "best_model.pt",
    splits_csv=BLACK_BOX_CHECKPOINT_DIR / "splits.csv",
    output_dir=BLACK_BOX_EMBEDDING_DIR,
    batch_size=256,
    overwrite=False
)

blackbox_embedding_paths

## 9. Prototype Clustering Grid Search

Fit K-means on the training split only, with K clusters per emotion. Each centroid is mapped to the nearest real training sample of the same emotion, and these real prototypes are used for classification. Select K and top-N on the validation split.

In [ ]:
from src.data.iemocap import EMOTION_NAMES
from src.training.train_prototype_clustering import (
    PrototypeClusteringTrainingConfig,
    train_prototype_clustering
)

prototype_training_config = PrototypeClusteringTrainingConfig(
    embedding_dim=128,
    num_classes=len(EMOTION_NAMES),
    cluster_counts=(1, 2, 3, 4, 5),
    top_ns=(1, 3, 5, 7, 9),
    monitor_metric="macro_f1",
    random_state=RANDOM_STATE
)

prototype_training_results = train_prototype_clustering(
    embedding_dir=BLACK_BOX_EMBEDDING_DIR,
    output_dir=PROTOTYPE_CLUSTERING_CHECKPOINT_DIR,
    config=prototype_training_config
)

## 10. Evaluate Prototype Clustering Classifier

In [ ]:
from src.evaluation.evaluate_prototype_clustering import evaluate_prototype_clustering
from src.evaluation.metrics import print_classification_metrics

prototype_test_metrics = evaluate_prototype_clustering(
    model_dir=PROTOTYPE_CLUSTERING_CHECKPOINT_DIR,
    embedding_dir=BLACK_BOX_EMBEDDING_DIR,
    split="test",
    output_dir=PROTOTYPE_CLUSTERING_REPORT_DIR
)

print_classification_metrics(prototype_test_metrics)

## 11. Evaluate Prototype Surrogate Fidelity

Measure how often the clustering surrogate matches the black-box predictions on the test split.

In [ ]:
from src.explainability.surrogate_fidelity import print_clustering_surrogate_fidelity_accuracy

surrogate_fidelity_metrics = print_clustering_surrogate_fidelity_accuracy(
    blackbox_checkpoint_path=BLACK_BOX_CHECKPOINT_DIR / "best_model.pt",
    feature_dir=FEATURE_DIR,
    prototype_model_dir=PROTOTYPE_CLUSTERING_CHECKPOINT_DIR,
    embedding_dir=BLACK_BOX_EMBEDDING_DIR,
    splits_csv=BLACK_BOX_CHECKPOINT_DIR / "splits.csv",
    split="test"
)

In [ ]:
from IPython.display import Image, display

prototype_confusion_matrix_path = PROTOTYPE_CLUSTERING_REPORT_DIR / "test_confusion_matrix.png"
if prototype_confusion_matrix_path.exists():
    display(Image(filename=str(prototype_confusion_matrix_path)))

## 12. Visualize Prototype Embedding Space

Project the saved 128-dimensional L2-normalized black-box embeddings to two PCA dimensions and highlight the prototypes of each emotion.

In [ ]:
from src.utils.visualize import plot_prototype_embedding_pca

prototype_pca_result = plot_prototype_embedding_pca(
    embedding_dir=BLACK_BOX_EMBEDDING_DIR,
    model_dir=PROTOTYPE_CLUSTERING_CHECKPOINT_DIR,
    split="all",
    output_path=PROTOTYPE_CLUSTERING_REPORT_DIR / "all_prototype_embedding_pca.png",
    random_state=RANDOM_STATE
)

prototype_pca_result["output_path"]

## 13. Explanation By Example - Inspect Prototype Neighbors of a Test Sample

Inspect one sample using the saved 128D embeddings and saved real prototypes. This does not reprocess the WAV file.

In [ ]:
import pandas as pd
from src.explainability.prototype_neighbors import (
    explain_sample_by_filename,
    print_prototype_explanation
)

# Set an IEMOCAP file name here, or leave it as None to use the first test sample.
SAMPLE_TO_EXPLAIN = None

prototype_explanation = explain_sample_by_filename(
    embedding_metadata=pd.read_csv(BLACK_BOX_EMBEDDING_DIR / "metadata.csv"),
    sample_to_explain=SAMPLE_TO_EXPLAIN,
    model_dir=PROTOTYPE_CLUSTERING_CHECKPOINT_DIR,
    embedding_dir=BLACK_BOX_EMBEDDING_DIR
)

print_prototype_explanation(prototype_explanation)

We can listen to the test sample and its nearest prototypes (medoids).

In [ ]:
from IPython.display import Audio, display, Markdown

sample_file = prototype_explanation["file_name"]
display(Markdown(f"### Sample: `{sample_file}`"))
display(Audio(filename=str(AUDIO_DIR / sample_file)))

display(Markdown("### Top prototypes"))
for prototype in prototype_explanation["top_prototypes"]:
    prototype_file = prototype.get("prototype_file_name")

    display(Markdown(
        f"**#{prototype['rank']} | {prototype['prototype_emotion']} | "
        f"similarity: {prototype['similarity']:.4f}**  \n"
        f"`{prototype_file}`"
    ))
    display(Audio(filename=str(AUDIO_DIR / prototype_file)))